In [ ]:
import sys
from pathlib import Path

# Add repo root to Python path
repo_root = Path("..").resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

In [ ]:
import pandas as pd
from pathlib import Path
from src.data import load_raw_csv
from src.prep import prepare_complaints_df, stratified_cap_per_class

TEXT_COL = "narrative"
LABEL_COL = "Product"
DATE_COL  = "Date received"

df = load_raw_csv("../data/raw")
df = prepare_complaints_df(df, text_col=TEXT_COL, label_col=LABEL_COL, date_col=DATE_COL, min_chars=20)

top_n = 10
top_labels = df[LABEL_COL].value_counts().head(top_n).index
df = df[df[LABEL_COL].isin(top_labels)].copy()

CAP_PER_CLASS = 10000
SEED = 42
df_small = stratified_cap_per_class(df, label_col=LABEL_COL, cap_per_class=CAP_PER_CLASS, random_state=SEED)

print(df_small.shape)
print(df_small[LABEL_COL].value_counts())

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

model_name = "all-MiniLM-L6-v2"
embedder = SentenceTransformer(model_name)

texts = df_small["text_clean"].tolist()

emb = embedder.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
).astype("float32")

emb.shape

In [ ]:
from sklearn.cluster import MiniBatchKMeans

K = 30
kmeans = MiniBatchKMeans(
    n_clusters=K,
    batch_size=4096,
    random_state=SEED,
    n_init="auto"
)

cluster_id = kmeans.fit_predict(emb)
df_small["cluster_id"] = cluster_id

df_small["cluster_id"].value_counts().head()